In [2]:
import pandas as pd

# Load dataset
data = pd.read_csv("recommender/Coursera.csv.zip")

# Perform the same cleaning steps you used before
# ...

# Create tags
data["tags"] = (
    data["Course Name"] +
    data["Difficulty Level"] +
    data["Course Description"] +
    data["Skills"]
)

# Create new_df
new_df = data[["Course Name", "Course URL", "tags"]].copy()

new_df.rename(columns={
    "Course Name": "course_name",
    "Course URL": "course_url"
}, inplace=True)

In [3]:
from sentence_transformers import SentenceTransformer

# Load a pretrained sentence embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for all course tags
embeddings = model.encode(
    new_df["tags"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 111/111 [05:08<00:00,  2.78s/it]


In [4]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(3522, 384)


In [5]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])

index.add(embeddings)

In [6]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(3522, 384)


In [7]:
import faiss
import numpy as np

# Convert embeddings to float32 (required by FAISS)
embeddings = np.array(embeddings).astype("float32")

# Create the FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])

# Add all course embeddings
index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 3522


In [9]:
def recommend(course_name):

    # 1. Get the index of the course
    course_index = new_df[new_df["course_name"] == course_name].index[0]

    # 2. Get embedding of that course
    query_vector = embeddings[course_index].reshape(1, -1)

    # 3. Search in FAISS index
    distances, indices = index.search(query_vector, 6)

    # 4. Build results
    recommendations = []

    for i in indices[0]:
        recommendations.append({
            "course_name": new_df.iloc[i]["course_name"],
            "course_url": new_df.iloc[i]["course_url"]
        })

    return recommendations

In [10]:
recommend("Finance for Managers")

[{'course_name': 'Finance for Managers',
  'course_url': 'https://www.coursera.org/learn/operational-finance'},
 {'course_name': 'Finance for Non-Financial Managers',
  'course_url': 'https://www.coursera.org/learn/finance-for-non-financial-managers'},
 {'course_name': 'Management and financial accounting: Know your numbers 1',
  'course_url': 'https://www.coursera.org/learn/management-accounting'},
 {'course_name': 'Corporate finance: Know your numbers 2',
  'course_url': 'https://www.coursera.org/learn/corporate-finance-know-your-numbers-2'},
 {'course_name': 'Fundamentals of financial and management accounting',
  'course_url': 'https://www.coursera.org/learn/financial-accounting-polimi'},
 {'course_name': 'Accounting for Decision Making',
  'course_url': 'https://www.coursera.org/learn/accounting'}]

In [11]:
import pickle
import faiss

# save embeddings
pickle.dump(embeddings, open("recommender/embeddings.pkl", "wb"))

# save dataframe
pickle.dump(new_df, open("recommender/new_df.pkl", "wb"))

# save FAISS index
faiss.write_index(index, "recommender/course.index")

In [13]:
#FAISS + embeddings + semantic search

In [14]:
import pickle

data = pickle.load(open("recommender/new_df.pkl", "rb"))

print(type(data))
print(len(data))
print(data.head())

<class 'pandas.DataFrame'>
3522
                                         course_name  \
0  Write A Feature Length Screenplay For Film Or ...   
1  Business Strategy: Business Model Canvas Analy...   
2                      Silicon Thin Film Solar Cells   
3                               Finance for Managers   
4       Retrieve Data using Single-Table SQL Queries   

                                          course_url  \
0  https://www.coursera.org/learn/write-a-feature...   
1  https://www.coursera.org/learn/canvas-analysis...   
2  https://www.coursera.org/learn/silicon-thin-fi...   
3  https://www.coursera.org/learn/operational-fin...   
4  https://www.coursera.org/learn/single-table-sq...   

                                                tags  
0  Write A Feature Length Screenplay For Film Or ...  
1  Business Strategy: Business Model Canvas Analy...  
2  Silicon Thin Film Solar CellsAdvancedThis cour...  
3  Finance for ManagersIntermediateWhen it comes ...  
4  Retrieve Data us